In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("phiard/aksara-jawa", output_dir='./data')

print("Path to dataset files:", path)

Path to dataset files: ./data


In [ ]:
import cv2
import os
import numpy as np
from pathlib import Path

# --- FUNGSI 1: Automatic Cropping ---
def automatic_crop(img, margin=5):
    """
    Melakukan crop otomatis untuk menghilangkan background putih di sekitar aksara.
    """
    # Ubah gambar ke Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Thresholding: Buat background jadi hitam murni dan tulisan jadi putih murni
    # Kita gunakan THRESH_BINARY_INV karena aksara aslinya hitam (nilai rendah)
    _, thresh = cv2.threshold(gray, 220, 255, cv2.THRESH_BINARY_INV)

    # Temukan koordinat piksel yang merupakan tulisan (warna putih di thresh)
    coords = cv2.findNonZero(thresh)

    if coords is None:
        # Jika gambar kosong/putih semua, kembalikan gambar asli
        return img

    # Dapatkan bounding box (x, y, w, h) dari tulisan
    x, y, w, h = cv2.boundingRect(coords)

    # Tambahkan sedikit margin agar tulisan tidak terlalu mepet ke tepi setelah di-crop
    y_start = max(0, y - margin)
    y_end = min(img.shape[0], y + h + margin)
    x_start = max(0, x - margin)
    x_end = min(img.shape[1], x + w + margin)

    # Lakukan cropping pada gambar asli
    cropped_img = img[y_start:y_end, x_start:x_end]

    return cropped_img

# --- FUNGSI 2: Resize dengan Padding (Letterboxing) ---
def resize_and_pad(img, target_size=128):
    """
    Resize gambar tanpa distorsi (menjaga aspect ratio)
    dan menambahkan padding putih agar ukuran menjadi persegi sempurna.
    """
    # Dapatkan dimensi gambar hasil crop (height, width)
    old_size = img.shape[:2]

    # Hitung rasio untuk resize berdasarkan sisi terpanjang
    ratio = float(target_size) / max(old_size)
    new_size = tuple([int(x * ratio) for x in old_size])

    # Resize gambar dengan mempertahankan aspect ratio asli
    img_resized = cv2.resize(img, (new_size[1], new_size[0]))

    # Hitung padding putih yang dibutuhkan (top, bottom, left, right)
    # untuk membuat gambar persegi sempurna
    delta_w = target_size - new_size[1]
    delta_h = target_size - new_size[0]

    # Bagi padding secara merata (center-aligned)
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)

    # Tentukan warna padding (Putih = 255, 255, 255 dalam BGR)
    color = [255, 255, 255]

    # Tambahkan border/padding murni putih ke sekeliling gambar
    final_img = cv2.copyMakeBorder(img_resized, top, bottom, left, right,
                                   cv2.BORDER_CONSTANT, value=color)

    return final_img

# --- FUNGSI UTAMA: Pipeline Pemrosesan Dataset ---
def process_complete_dataset(input_base_dir, output_base_dir, target_size=128):
    """
    Pipeline lengkap untuk membaca, meng-crop, me-resize+pad, dan menyimpan dataset.
    """
    input_base = Path(input_base_dir)
    output_base = Path(output_base_dir)

    # Daftar folder utama yang ingin diproses
    sub_dirs = ['train', 'val']

    total_processed = 0
    total_errors = 0

    print("="*60)
    print(f"Memulai Pemrosesan Dataset Aksara Jawa")
    print(f"Target Ukuran Akhir: {target_size}x{target_size} piksel")
    print("="*60)

    for sub_dir in sub_dirs:
        input_path = input_base / sub_dir
        output_path = output_base / sub_dir

        # Mengecek apakah folder input ada
        if not input_path.exists():
            print(f"❌ Folder {input_path} tidak ditemukan, dilewati.")
            continue

        print(f"\n📂 Memproses direktori: {sub_dir}")

        # Looping melalui setiap subfolder kelas aksara (ba, ca, da, dst)
        # sorted() memastikan urutan alfabetis
        for class_path in sorted(input_path.iterdir()):
            if not class_path.is_dir():
                continue

            class_name = class_path.name
            class_output_path = output_path / class_name

            # Buat folder output kelas jika belum ada
            class_output_path.mkdir(parents=True, exist_ok=True)

            # Looping melalui semua gambar di dalam folder aksara
            count = 0
            for img_path in class_path.iterdir():
                # Filter hanya file gambar yang didukung
                if img_path.suffix.lower() not in ['.png', '.jpg', '.jpeg']:
                    continue

                # 1. BACA GAMBAR
                # Menggunakan Path str() agar kompatibel dengan cv2
                img = cv2.imread(str(img_path))

                if img is None:
                    print(f"   ⚠️ Gagal membaca gambar: {img_path.name}")
                    total_errors += 1
                    continue

                # --- PIPELINE PEMROSESAN GAMBAR ---

                # Langkah A: Crop Otomatis (Hapus White Space Berlebih)
                img_cropped = automatic_crop(img, margin=5)

                # Langkah B: Resize dengan Padding Putih (Seragamkan Ukuran)
                final_img = resize_and_pad(img_cropped, target_size=target_size)

                # Langkah C: SIMPAN GAMBAR
                save_path = class_output_path / img_path.name
                cv2.imwrite(str(save_path), final_img)

                count += 1
                total_processed += 1

            print(f"   ✅ Selesai memproses kelas '{class_name}': {count} gambar.")

    print("\n" + "="*60)
    print(f"📊 RINGKASAN PEMROSESAN:")
    print(f"   Total gambar berhasil diproses: {total_processed}")
    print(f"   Total kegagalan membaca gambar: {total_errors}")
    print(f"   Output disimpan di: {output_base_dir}")
    print("="*60)

# --- KONFIGURASI DAN EKSEKUSI ---
if __name__ == "__main__":
    # 1. Konfigurasi Direktori Input (Sesuai dengan struktur pohon Anda)
    # Pastikan folder 'data' ada di direktori yang sama dengan script ini
    INPUT_DATA_DIR = "data/v3/v3"

    # 2. Konfigurasi Direktori Output Aman (Agar data asli tidak tertimpa)
    # Anda bisa menamainya 'data_uniform' atau 'data_for_model'
    OUTPUT_DATA_DIR = "data/final"

    # 3. Target Ukuran
    FINAL_RESOLUTION = 224

    # Jalankan proses
    process_complete_dataset(INPUT_DATA_DIR, OUTPUT_DATA_DIR, FINAL_RESOLUTION)

Memulai Pemrosesan Dataset Aksara Jawa
Target Ukuran Akhir: 224x224 piksel

📂 Memproses direktori: train
   ✅ Selesai memproses kelas 'ba': 114 gambar.
   ✅ Selesai memproses kelas 'ca': 108 gambar.
   ✅ Selesai memproses kelas 'da': 108 gambar.
   ✅ Selesai memproses kelas 'dha': 108 gambar.
   ✅ Selesai memproses kelas 'ga': 108 gambar.
   ✅ Selesai memproses kelas 'ha': 102 gambar.
   ✅ Selesai memproses kelas 'ja': 108 gambar.
   ✅ Selesai memproses kelas 'ka': 108 gambar.
   ✅ Selesai memproses kelas 'la': 108 gambar.
   ✅ Selesai memproses kelas 'ma': 108 gambar.
   ✅ Selesai memproses kelas 'na': 108 gambar.
   ✅ Selesai memproses kelas 'nga': 102 gambar.
   ✅ Selesai memproses kelas 'nya': 108 gambar.
   ✅ Selesai memproses kelas 'pa': 108 gambar.
   ✅ Selesai memproses kelas 'ra': 108 gambar.
   ✅ Selesai memproses kelas 'sa': 108 gambar.
   ✅ Selesai memproses kelas 'ta': 108 gambar.
   ✅ Selesai memproses kelas 'tha': 108 gambar.
   ✅ Selesai memproses kelas 'wa': 108 gambar

In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26n-cls.yaml")  # build a new model from YAML
model = YOLO("yolo26n-cls.pt")  # load a pretrained model (recommended for training)
model = YOLO("yolo26n-cls.yaml").load("yolo26n-cls.pt")  # build from YAML and transfer weights

# Train the model
results = model.train(data="mnist160", epochs=100, imgsz=64)